In [1]:
%pip install --force-reinstall --no-deps git+https://github.com/chrisjcameron/TexSoup.git@mixed-args
#! pip install --editable /Users/cjc73/gits/arxiv/TexSoup/

  Cloning https://github.com/chrisjcameron/TexSoup.git (to revision mixed-args) to /private/var/folders/b4/3gwt7cpj48n9_chzk5570rc80000gp/T/pip-req-build-gakturmx
  Running command git clone --filter=blob:none --quiet https://github.com/chrisjcameron/TexSoup.git /private/var/folders/b4/3gwt7cpj48n9_chzk5570rc80000gp/T/pip-req-build-gakturmx
  Running command git checkout -b mixed-args --track origin/mixed-args
  Switched to a new branch 'mixed-args'
  branch 'mixed-args' set up to track 'origin/mixed-args'.
  Resolved https://github.com/chrisjcameron/TexSoup.git to commit a5f0d18d89bf0df21f0b47d4bea83dbdabfa93e9
  Preparing metadata (setup.py) ... done
  Created wheel for TexSoup: filename=TexSoup-0.3.1-py3-none-any.whl size=32722 sha256=ccd7199ba293b6a5fc52da81f890fb5455510d46b35a6184969afdd05fcfe3f8
  Stored in directory: /private/var/folders/b4/3gwt7cpj48n9_chzk5570rc80000gp/T/pip-ephem-wheel-cache-5_zyae1w/wheels/ba/c5/3d/85bc5f5e9a69dfe1de6d94bc3252986de5dce0456aa8f48a58
Successfu

In [12]:
#import zipfile
import tarfile
import io
import importlib
import os
import regex as re
import glob
import pandas as pd
import itertools as itr
import pprint

In [13]:
import pyperclip   #copy text to clipboard for inspecting

In [14]:
from tqdm.auto import tqdm

In [15]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

In [16]:
import TexSoup as TS
from TexSoup.tokens import MATH_ENV_NAMES
# -

In [17]:
TS.__file__

'/Users/cjc73/miniconda3/envs/cforge/lib/python3.11/site-packages/TexSoup/__init__.py'

In [18]:
TS.__file__

'/Users/cjc73/miniconda3/envs/cforge/lib/python3.11/site-packages/TexSoup/__init__.py'

In [19]:
#importlib.reload(TS)

In [20]:
LOCAL_DATA_PATH = './data/2201_00_all/'

In [21]:
def pre_format(text):
    '''Apply some substititions to make LaTeX easier to parse'''
    source_text = (
        text
        .replace('\\}\\', '\\} \\')  # Due to escape rules \\ is equivalent to \
        .replace(')}', ') }')
        .replace(')$', ') $')
        #.replace(r'\left [', r'\left[ ')
        #.replace(r'\left (', r'\left( ')
        #.replace(r'\left \{', r'\left\{ ')
    )
    return source_text
    #clean_lines = []
    #for line in source_text.splitlines(False):
    #    cleanline = line.strip()
    #    if cleanline.startswith(r'\newcommand'):
    #        cleanline = r'%' + cleanline
    #    elif cleanline.startswith(r'\def'):
    #        cleanline = r'%' + cleanline
    #    clean_lines.append(cleanline)
    #return '\n'.join(clean_lines)

In [22]:
def find_doc_class(wrapped_file, name_match=False):
    '''Search for document class related lines in a file  and return a code to represent the type'''
    doc_class_pat = re.compile(r"^\s*\\document(?:style|class)")
    sub_doc_class = re.compile(r"^\s*\\document(?:style|class).*(?:\{standalone\}|\{subfiles\})")

    for line in wrapped_file:
        if doc_class_pat.search(line):
            if name_match:
                # we can miss if there are two or more lines with documentclass 
                # and the first one is not the one that has standalone/subfile
                if sub_doc_class.search(line):
                    return -99999
                return 1 #main_files[tf] = 1
            
    return 0 #main_files[tf] = 0


def find_main_tex_source_in_tar(tar_path, encoding='utf-8'):
    '''Identify the main Tex file in a tarfile.
    
    Args:
        tar_path: A gzipped tar archive of a directory containing tex source and support files.
    '''
    
    tex_names = set(["paper", "main", "ms.", "article"])

    with tarfile.open(tar_path, 'r') as in_tar:
        tex_files = [f for f in in_tar.getnames() if f.endswith('.tex')]
        
        # got one file
        if len(tex_files) == 1:
            return tex_files[0]

        main_files = {}
        for tf in tex_files:
            depth = len(tf.split('/')) - 1
            has_main_name = any(kw in tf for kw in tex_names)
            fp = in_tar.extractfile(tf)
            wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=encoding) #universal newlines
            # does it have a doc class?
            # get the type
            main_files[tf] = find_doc_class(wrapped_file, name_match = has_main_name) - depth 
            wrapped_file.close() 
        
        # got one file with doc class
        if len(main_files) == 1:
            return(main_files.keys()[0])
        
        # account for multi-file submissions
        return(max(main_files, key=main_files.get))

In [23]:
def soup_from_tar(tar_path, encoding='utf-8', tolerance=0):
    tex_main = find_main_tex_source_in_tar(tar_path, encoding=encoding)
    with tarfile.open(tar_path, 'r') as in_tar:
        fp = in_tar.extractfile(tex_main)
        wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=encoding) #universal newlines
        source_text = pre_format(wrapped_file.read())
        soup = TS.TexSoup(source_text, tolerance=tolerance, skip_envs=MATH_ENV_NAMES)
        return soup

In [24]:
def source_from_tar(tar_path, encoding='utf-8'):
    tex_main = find_main_tex_source_in_tar(tar_path, encoding=encoding)
    with tarfile.open(tar_path, 'r') as in_tar:
        fp = in_tar.extractfile(tex_main)
        wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=encoding) #universal newlines
        source_text = pre_format(wrapped_file.read())
        return source_text

In [25]:
swap = itr.cycle((True, False))

def find_bad(current_text_lines):
    mid = int(len(current_text_lines)/2)
    part_a = current_text_lines[0:mid]
    part_b = current_text_lines[mid:]
    if next(swap):
        part_b, part_a = part_a, part_b
    bad = ""
    try:
        soup = TS.TexSoup("\n".join(part_a), tolerance=tolerance, skip_envs=MATH_ENV_NAMES)
    except KeyboardInterrupt:
        raise
    except:
        return part_a
    try:
        soup = TS.TexSoup("\n".join(part_b), tolerance=tolerance, skip_envs=MATH_ENV_NAMES)
    except KeyboardInterrupt:
        raise
    except:
        return part_b
    return "--"
    

def find_bad_lines(tar_path, encoding='utf-8'):
    tex_main = find_main_tex_source_in_tar(tar_path, encoding=encoding)
    with tarfile.open(tar_path, 'r') as in_tar:
        fp = in_tar.extractfile(tex_main)
        wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=encoding) #universal newlines
        source_text = pre_format(wrapped_file.read())
        current_text = source_text.splitlines()

    while len(current_text) > 1:
        bad_half = find_bad(current_text)
        if current_text == bad_half:
            break
        current_text = bad_half
        
    return bad_half

In [26]:
def show_context(text_path, offset, context_size=50):
    try: 
        with open(text_path, 'r', encoding='utf-8') as file:
            file.seek(offset)
            context = file.read(context_size)
            return context
            # Is it unicode?
    except UnicodeDecodeError as ue:
        pass
    try:
        with open(text_path, 'r', encoding='latin-1') as file:
            file.seek(offset)
            context = file.read(context_size)
            return context
    except: 
        raise

# file_path = 'Ising_v2.tex'
# offset_position = 805
# context = show_context(file_path, offset_position)
# print("Error context at offset 805:", context)

In [27]:
#show_context(infile_path, 8584)

## Check a file with parse errors

In [32]:
min_example=r"""
\newcounter{savenumi}
\newenvironment{savenumerate}{\begin{enumerate}
\setcounter{enumi}{\value{savenumi}}}{\end{enumerate}
\setcounter{savenumi}{\value{enumi}}}
\newtheorem{theoremfoo}{Theorem}[section] %by chapter in report style
\newenvironment{theorem}{\pagebreak[1]\begin{theoremfoo}}{\end{theoremfoo}}
\newenvironment{repeatedtheorem}[1]{\vskip 6pt
\noindent
{\bf Theorem #1}\ \em
}{}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
tsoup = TS.TexSoup(min_example, tolerance=0)
print(tsoup)
#print(min_example)
for item in tsoup.all:
    print("-----")
    print(item)

\newcounter{savenumi}
\newenvironment{savenumerate}{\begin{enumerate}
\setcounter{enumi}{\value{savenumi}}}{\end{enumerate}
\setcounter{savenumi}{\value{enumi}}}\newtheorem{theoremfoo}{Theorem}[section] %by chapter in report style
\newenvironment{theorem}{\pagebreak[1]\begin{theoremfoo}}{\end{theoremfoo}}
\newenvironment{repeatedtheorem}[1]{\vskip 6pt
\noindent
{\bf Theorem #1}\ \em
}{}
-----
\newcounter{savenumi}
-----


-----
\newenvironment{savenumerate}{\begin{enumerate}
\setcounter{enumi}{\value{savenumi}}}{\end{enumerate}
\setcounter{savenumi}{\value{enumi}}}\newtheorem
-----
{theoremfoo}
-----
{Theorem}
-----
[
-----
section
-----
]
-----
 
-----
%by chapter in report style
-----


-----
\newenvironment{theorem}{\pagebreak[1]\begin{theoremfoo}}{\end{theoremfoo}}
-----


-----
\newenvironment{repeatedtheorem}[1]{\vskip 6pt
\noindent
{\bf Theorem #1}\ \em
}{}


In [17]:
min_example=r"""
$\braket{\mathcal N_N^\ell}$

""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

$\braket{\mathcal N_N^\ell}$

In [18]:
min_example=r"""
$$ \ceil[\Big]{\frac{foo}{bar}} $$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

$$ \ceil[\Big]{\frac{foo}{bar}} $$

In [19]:
min_example=r"""
\begin{abstract}
    Foo. Bar. Foo.
\end {abstract}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
try:
    TS.TexSoup(pre_format(min_example), tolerance=0)
except AssertionError as e:
    print(e)
#print(min_example)

\begin{abstract}
    Foo. Bar. Foo.
\end{abstract}

In [20]:
min_example=r"""
$\mat{\MsI}{x,\cX}$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

$\mat{\MsI}{x,\cX}$

In [21]:
min_example=r"""

$\braket{\tau }$,


""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

# The problem the new* changing how to interpret the \begin

$\braket{\tau }$,

In [22]:
min_example=r"""
\begin {abstract}
    Foo. Bar. Foo.
\end {abstract}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
try:
    TS.TexSoup(pre_format(min_example), tolerance=0)
except AssertionError as e:
    print(e)
#print(min_example)

\begin{abstract}
    Foo. Bar. Foo.
\end{abstract}

In [23]:
min_example=r"""
\newenvironment{rep#1}[1]{%
 \def\rep@title{#2 \ref{##1}}%
 \begin{rep@theorem}}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

# The problem the new* changing how to interpret the \begin

\newenvironment{rep#1}[1]{%
 \def\rep{@}{#2 \ref{##1}}%
 \begin{rep@theorem}}

In [24]:
min_example=r"""
{\subsection}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
try:
    TS.TexSoup(pre_format(min_example), tolerance=0)
except AssertionError as e:
    print(e)
#print(min_example)

{\subsection}

In [25]:
min_example=r"""
\renewcommand{\tilde}{\widetilde}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
try:
    TS.TexSoup(pre_format(min_example), tolerance=0)
except AssertionError as e:
    print(e)
#print(min_example)

\renewcommand{\tilde}{\widetilde}

In [26]:
infile_path = "./data/2201_00_all/2201.00001v1.tar.gz" #'./data/2201_samp/2201.00048v1.tar.gz'

text = source_from_tar(infile_path)
pyperclip.copy(text)
soup = soup_from_tar(infile_path, tolerance=1)


title = soup.find('title')
if title: print(f"{title.name}: {title.text}")
for sec in soup.find_all('section'):
    print(f' {sec.name}: {sec.text}')

title: ['Modeling Advection on Directed Graphs using  Mat', "\\'", 'e', 'rn Gaussian Processes for Traffic Flow']
 section: ['Introduction']
 section: ['Understanding the directed graph advection operator']
 section: ['Directed Graph Advection Mat', "\\'", 'e', 'rn Gaussian Process (DGAMGP) ']
 section: ['Numerical Results']
 section: ['Conclusions']
 section: ['Upwinding discretizations of linear advection']
 section: ['Examples of ', '$', 'L_', 'adv', '$', ' on balanced graphs resulting in finite difference discretizations of linear advection']
 section: ['Additional Experiments']


In [27]:
min_example=r"""
\author*[4,5]{\fnm{Honghao} \sur{Gao}}\email{honghaogao@gachon.ac.kr; gaohonghao@shu.edu.cn}
""".strip()
tsoup = TS.TexSoup(min_example, tolerance=0)
print(tsoup)

\author*[4,5]{\fnm{Honghao} \sur{Gao}}\email{honghaogao@gachon.ac.kr; gaohonghao@shu.edu.cn}


In [28]:
min_example=r"""

\newcommand{\bq}{\begin{eqnarray*}}
\newcommand{\eq}{\end{eqnarray*}}
\newcommand{\bqn}{\begin{eqnarray}}
\newcommand{\eqn}{\end{eqnarray}}

\newcommand{\defproblem}[3]{
 \vspace{1mm}
\noindent\fbox{
 \begin{minipage}{0.96\textwidth}
 {\bf{Input:}} #2 \\
 {\bf{Question:}} #3
 \end{minipage}
 }
 \vspace{1mm}
}

""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

\newcommand{\bq}{\begin{eqnarray*}}
\newcommand{\eq}{\end{eqnarray*}}
\newcommand{\bqn}{\begin{eqnarray}}
\newcommand{\eqn}{\end{eqnarray}}

\newcommand{\defproblem}[3]{
 \vspace{1mm}
\noindent\fbox{
 \begin{minipage}{0.96\textwidth}
 {\bf{Input:}} #2 \\
 {\bf{Question:}} #3
 \end{minipage}
 }
 \vspace{1mm}
}

In [29]:
min_example=r"""

\newcommand{\bq}{\begin{eqnarray*}}
\newcommand{\eq}{\end{eqnarray*}}

\begin{theorem} 
\bq \mathbb{E} = \mathcal{X} \eq
\end{theorem}


""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

\newcommand{\bq}{\begin{eqnarray*}}
\newcommand{\eq}{\end{eqnarray*}}

\begin{theorem} 
\bq \mathbb{E} = \mathcal{X} \eq
\end{theorem}

In [30]:
min_example=r"""
\def\fc#1#2{\frac{#1}{#2}}
\def\h{\frac{1}{2}}
\newcommand{\nwc}{\newcommand}
\nwc{\ba}  {\begin{array}}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

\def\fc#1#2{\frac{#1}{#2}}
\def\h{\frac{1}{2}}
\newcommand{\nwc}{\newcommand}
\nwc{\ba}{\begin{array}}

In [31]:
min_example=r"""
\def\lf{\left}
""".strip()
tsoup = TS.TexSoup(min_example, tolerance=0)
print(tsoup)

\def\lf{\left}


In [32]:
min_example=r"""
 \def\be   {\begin{equation}} \def\ee   {\end{equation}}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
soup = TS.TexSoup(pre_format(min_example), tolerance=0)
print(soup)
#print(min_example)

\def\be{\begin{equation}} \def\ee{\end{equation}}


In [33]:
min_example=r"""
\newcommand{\repeatargs}[2]{%
  First argument: #1 \\
  Second argument: #2 \\
  Repeating first argument: #1 \\
  Repeating second argument: #2
}

""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
soup = TS.TexSoup(pre_format(min_example), tolerance=0)
print(soup)
#print(min_example)

\newcommand{\repeatargs}[2]{%
  First argument: #1 \\
  Second argument: #2 \\
  Repeating first argument: #1 \\
  Repeating second argument: #2
}


## Quick check a folder of tar files

In [34]:
files = glob.glob(f'{LOCAL_DATA_PATH}/*.tar.gz')
files_count = len(files)
utf_count = 0
latin_count = 0 
err_files = {}

TOLERANCE = 1

with tqdm(total=files_count, desc="errors") as err_prog:
    for i, tar_file in enumerate(tqdm(files, desc="Progress", display=True)):
        # Is it unicode?
        try:
            soup = soup_from_tar(tar_file, encoding='utf-8', tolerance=TOLERANCE)
            utf_count += 1
            continue
        except EOFError as eof:
            err_files[(i, tar_file)] = type(eof)
            _ = err_prog.update(1)
            continue
        except UnicodeDecodeError as ue:
            pass
        except KeyboardInterrupt as KB_err:
            break
        except Exception as e:
            err_files[(i, tar_file)] = type(e)
            _ = err_prog.update(1)
            continue

        # Is it something else?
        try:
            soup = soup_from_tar(tar_file, encoding='latin-1', tolerance=TOLERANCE)
            latin_count += 1
            continue
        except KeyboardInterrupt as KB_err:
            break
        except Exception as e:
            err_files[(i, tar_file)] = type(e)
            _ = err_prog.update(1)
            pass

errors:   0%|          | 0/405 [00:00<?, ?it/s]

Progress:   0%|          | 0/405 [00:00<?, ?it/s]

In [35]:
REAL_ERROR_FILES = {
    './data/2201_00_all/2201.00683v2.tar.gz': "{\cite{faure2017semiclassical}.", 
    './data/2201_00_all/2201.00683v1.tar.gz': "{\cite{faure2017semiclassical}",
    './data/2201_00_all/2201.00718v1.tar.gz': "bad main file identification", 
    './data/2201_00_all/2201.00907v1.tar.gz': "bad main file identification", 
} 

In [36]:
print(f"{files_count} processed, {len(err_files)} failures.")
print(f"UTF8: {utf_count}; Latin1: {latin_count}")
len(err_files)
for k,v in err_files.items():
    prefix = ""
    if k[1] in REAL_ERROR_FILES:
        prefix = "*"
    print(f"{prefix} {k[0]} - {k[1]}: {v}")

405 processed, 4 failures.
UTF8: 397; Latin1: 4


4

* 88 - ./data/2201_00_all/2201.00683v2.tar.gz: <class 'AttributeError'>
* 239 - ./data/2201_00_all/2201.00718v1.tar.gz: <class 'ValueError'>
* 241 - ./data/2201_00_all/2201.00683v1.tar.gz: <class 'AttributeError'>
* 360 - ./data/2201_00_all/2201.00907v1.tar.gz: <class 'ValueError'>


## Scratch below here

In [37]:
err_file_paths = list(y for (x,y) in err_files.keys() if y not in REAL_ERROR_FILES)

In [38]:
show_context(err_file_paths[1], 15322)

IndexError: list index out of range

In [ ]:
find_bad_lines(err_file_paths[1])

In [39]:
err_file_paths

[]

In [80]:
TOLERANCE = 1
encoding='utf-8'
#encoding='latin-1'
idx = 3
err_file_paths[idx]
infile_path = err_file_paths[idx] # "./data/2201_00_all/2201.00732v1.tar.gz"
#"/Volumes/Neptune/scratch/2404/2404.08812v1.tar.gz" #'./data/2201_samp/2201.00048v1.tar.gz'

text = source_from_tar(infile_path, encoding=encoding)
pyperclip.copy(text)
soup = soup_from_tar(infile_path, encoding=encoding, tolerance=TOLERANCE)


title = soup.find('title')
if title: print(f"{title.name}: {title.text}")
for sec in soup.find_all('section'):
    print(f' {sec.name}: {sec.text}')

'./data/2201_00_all/2201.00907v1.tar.gz'

ValueError: max() arg is an empty sequence

In [31]:
TOLERANCE = 0
encoding='utf-8'
#encoding='latin-1'
idx = 1
#err_file_paths[idx]
infile_path = files[230]
#"/Volumes/Neptune/scratch/2404/2404.08812v1.tar.gz" #'./data/2201_samp/2201.00048v1.tar.gz'

text = source_from_tar(infile_path, encoding=encoding)
pyperclip.copy(text)
soup = soup_from_tar(infile_path, encoding=encoding, tolerance=TOLERANCE)


title = soup.find('title')
if title: print(f"{title.name}: {title.text}")
for sec in soup.find_all('section'):
    print(f' {sec.name}: {sec.text}')

Cmd name not recognized in \newcommand{Noneromannumeral None}[1]{\romannumeral #1}.
Cmd name not recognized in \newcommand{Noneexpandafter\@slowromancap\romannumeral None@}[1]{\expandafter\@slowromancap\romannumeral #1@}.
Cmd name not recognized in \newcommand{None{}
\newcommand{Nonegamma}{\gamma}
\newcommand{None{\rm exp}}{r_{\rm exp}}
\defNonenonumber{\nonumber}
\def\mr{M_R}
\def\mk{m_K}
\def\gr{\Gamma_R}

\makeatother

\textwidth 500 pt
\textheight 710 pt
\hoffset -25 pt
\voffset -30 pt

\begin{document}


\title{Analysis on the composite  nature of the light scalar mesons $f_{0}(980) $ and $a_0(980) $}

\author{Ze-Qiang Wang$^1$}
\author{Xian-Wei Kang$^{1,2}$}\email{xwkang@bnu.edu.cn}
\author{J.~A.~Oller$^3$}\email{oller@um.es}
\author{Lu Zhang$^1$}

\affiliation{{$^{1}$ Key Laboratory of Beam Technology of Ministry of Education, College of Nuclear Science and Technology, Beijing Normal University, Beijing 100875, China\\
$^{2}$ Beijing Radiation Center, Beijing 100875, China\\
$^{

TypeError: [Line: 0, Offset 126566] Malformed argument. First and last elements must match a valid argument format. In this case, TexSoup could not find matching punctuation for: {.
Just finished parsing: ['{', '$', None, BraceGroup(TexCmd('rm'), ' exp'), '=0.85', TexCmd('pm'), ' 0.15', '$', '. We indeed find a strong sensitivity, such that  for the central value ', '$', None, BraceGroup(TexCmd('rm'), ' exp'), '=0.85', '$', ' we have the values for the couplings ', '$', '|', TexCmd('gamma'), '_1|=2.9', '$', '~GeV and ', '$', '|', TexCmd('gamma'), '_2|=2.2', '$', '~GeV.\nWithin one standard deviation the upper value ', '$', None, BraceGroup(TexCmd('rm'), ' exp'), '=0.85+0.14=0.99', '$', ' cannot be reproduced, that is, no solution for the Flatt', "\\'", 'e bare parameters is found in that case.\nFor the lower end ', '$', None, BraceGroup(TexCmd('rm'), ' exp'), '=0.85-0.15=0.70', '$', ' we find a new solution with the values\n', '$', '|', TexCmd('gamma'), '_1|=3.1', '$', '~GeV and ', '$'

In [ ]:
soup.find_all('section')

In [ ]:
soup

In [ ]:
find_bad_lines(infile_path, encoding='utf-8')

In [ ]:
tar_path = "./data/2201_samp/2201.00008v2.tar.gz"
encoding = "utf-8"
with tarfile.open(tar_path, 'r') as in_tar:
    tex_files = [f for f in in_tar.getnames() if f.endswith('.tex')]

    # got one file
    if len(tex_files) == 1:
        pass #return tex_files[0]

    main_files = {}
    for tf in tex_files:
        fp = in_tar.extractfile(tf)
        wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=encoding) #universal newlines
        # does it have a doc class?
        # get the type
        main_files[tf] = find_doc_class(wrapped_file)
        wrapped_file.close() 

    # got one file with doc class
    if len(main_files) == 1:
        pass #return(main_files.keys()[0])

    # account for multi-file submissions
    #return(max(main_files, key=main_files.get))

In [ ]:
main_files

In [ ]:
doc_class_pat = re.compile(r"^\s*\\document(?:style|class)")

with tarfile.open(tar_path, 'r', encoding='utf-8') as in_tar:
    #in_tar.getnames()
    fp = in_tar.extractfile('main.tex')
    wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8') #universal newlines
    for line in wrapped_file:
        if doc_class_pat.search(line):
            print(line)
            break

In [ ]:
next(wrapped_file)

In [ ]:
min_example=r"""
\documentclass{article}
\begin{document}
% \renewcommand{\shorttitle}{Avoiding Catastrophe}
\end{document}
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example))
#print(min_example)
TS.TexSoup(r'\newcommand{\bra}[1]{\left\langle#1\right|}')

In [ ]:
TS.TexSoup(r'\def\be{\foo{equation}}')

In [ ]:
TS.TexSoup(r'\renewcommand{\shorttitle}{Avoiding Catastrophe}')
min_example = r"\newenvironment{inlinemath}{$}{$}".strip()
TS.TexSoup(pre_format(min_example))
#print(min_example)

In [ ]:
min_example = r"In practice, the matrix $\left [ 4 \right]\Inv\M{D}^{(1)}_n $".strip()
TS.TexSoup(pre_format(min_example))
#print(min_example)

In [ ]:
min_example = r"In practice, the matrix $\left[ 4 \right]\Inv\M{D}^{(1)}_n $"


cats = TS.category.categorize(min_example)
tokens = list(TS.tokens.tokenize(cats))

char_codes = list(TS.category.categorize(min_example))

buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(r'\left[ 4 \right]')))
TS.reader.read_command(buf, n_required_args=-1, mode='mode:math', skip=1 )

In [37]:
min_example=r"""
\renewcommand{\subsection}[1]{{\textit{#1.~}}}""".strip()

cats = TS.category.categorize(min_example)
tokens = list(TS.tokens.tokenize(cats))

char_codes = list(TS.category.categorize(min_example))

with pd.option_context('display.max.columns', None, 'display.max_colwidth', 0):
    pd.DataFrame({'char':char_codes, 'code':(x.category for x in char_codes)}).transpose()
    pd.DataFrame({'tokens':tokens})

print("Cmd Math Mode")
buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
rd_cmd = TS.reader.read_command(buf, n_required_args=2, n_optional_args=2, mode='mode:math', tolerance=1)
pprint.pp(rd_cmd)

print("Cmd Non Math")
buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
rd_cmd = TS.reader.read_command(buf,  n_required_args=2, n_optional_args=2, tolerance=1)
pprint.pp(rd_cmd)

print("Reader")
buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
rd = TS.read(buf, tolerance=1)
pprint.pp(rd)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45
char,\,r,e,n,e,w,c,o,m,m,a,n,d,{,\,s,u,b,s,e,c,t,i,o,n,},[,1,],{,{,\,t,e,x,t,i,t,{,#,1,.,~,},},}
code,1,12,12,12,12,12,12,12,12,12,12,12,12,2,1,12,12,12,12,12,12,12,12,12,12,3,19,13,20,2,2,1,12,12,12,12,12,12,2,7,13,13,14,3,3,3


,tokens
0,\
1,renewcommand
2,{
3,\
4,subsection
5,}
6,[
7,1
8,]
9,{


Cmd Math Mode
('\\', [BraceGroup('r'), BraceGroup('e')])
Cmd Non Math
('\\', [BraceGroup('r'), BraceGroup('e')])
Reader
([TexCmd('renewcommand', [BraceGroup(TexCmd('subsection', [BraceGroup('}'), BracketGroup('1')]), BraceGroup(BraceGroup(TexCmd('textit', [BraceGroup('#1.~')]))))])],
 '\\renewcommand{\\subsection}[1]{{\\textit{#1.~}}}')


In [35]:
min_example = r"\subsection[Background Info]{Background}"


cats = TS.category.categorize(min_example)
tokens = list(TS.tokens.tokenize(cats))

char_codes = list(TS.category.categorize(min_example))

with pd.option_context('display.max.columns', None, 'display.max_colwidth', 0):
    pd.DataFrame({'char':char_codes, 'code':(x.category for x in char_codes)}).transpose()
    pd.DataFrame({'tokens':tokens})

buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
TS.reader.read_command(buf, n_required_args=-1, mode='mode:math', skip=3, tolerance=1)

buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
TS.read(buf, tolerance=1)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39
char,\,s,u,b,s,e,c,t,i,o,n,[,B,a,c,k,g,r,o,u,n,d,,I,n,f,o,],{,B,a,c,k,g,r,o,u,n,d,}
code,1,12,12,12,12,12,12,12,12,12,12,19,12,12,12,12,12,12,12,12,12,12,11,12,12,12,12,20,2,12,12,12,12,12,12,12,12,12,12,3


,tokens
0,\
1,subsection
2,[
3,Background Info
4,]
5,{
6,Background
7,}


('Background Info', [])

([TexCmd('subsection', [BracketGroup('Background Info'), BraceGroup('Background')])],
 '\\subsection[Background Info]{Background}')

In [29]:
min_example = r"$ t \in [0,1] $$ t \in [0,1] $"


cats = TS.category.categorize(min_example)
tokens = list(TS.tokens.tokenize(cats))

char_codes = list(TS.category.categorize(min_example))

with pd.option_context('display.max.columns', None, 'display.max_colwidth', 0):
    pd.DataFrame({'char':char_codes, 'code':(x.category for x in char_codes)}).transpose()
    pd.DataFrame({'tokens':tokens})

buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
TS.reader.read_command(buf, n_required_args=-1, mode='mode:math', skip=3, tolerance=1)

buf = TS.reader.Buffer(TS.tokens.tokenize(TS.category.categorize(min_example)))
TS.read(buf, tolerance=1)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
char,$,,t,,\,i,n,,[,0,",",1,],,$,$,,t,,\,i,n,,[,0,",",1,],,$
code,4,11,12,11,1,12,12,11,19,13,13,13,20,11,4,4,11,12,11,1,12,12,11,19,13,13,13,20,11,4


,tokens
0,$
1,t
2,\
3,in
4,
5,[
6,"0,1"
7,]
8,
9,$


('in', [])

EOFError: [Line: 0, Offset: 19] "$" env expecting $. Reached end of file.

In [ ]:
with pd.option_context('display.max.columns', None, 'display.max_colwidth', 0):
    pd.DataFrame({'char':char_codes, 'code':(x.category for x in char_codes)}).transpose()
    pd.DataFrame({'tokens':tokens})

In [ ]:
min_example = r"In practice, the matrix $\left [\M{D}^{(1)}_n(\M{D}^{(1)}_n)\Tra\right]\Inv\M{D}^{(1)}_n $"
print(min_example)
TS.TexSoup(pre_format(min_example))
TS.TexSoup(min_example)

In [ ]:
min_example=r"""
\documentclass{article}
\begin{document}
% \renewcommand{\shorttitle}{Avoiding Catastrophe}
\end{document}
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example))
#print(min_example)

In [ ]:
bmin_example=r"""
\def\bean {\begin{foo}}  \def\eean {\end{foo}}
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example))
TS.TexSoup(min_example)
print(min_example)
min_example=r"""
we {use $A=8B$ and $s=1$, then the scalar field becomes same with (\Ref{scalarfield}) and
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#TS.TexSoup(min_example)
print(min_example)
print(pre_format(min_example))
BRACKETS_DELIMITERS = {
    '(', ')', '<', '>', '[', ']', '{', '}', r'\{', r'\}', '.' '|', r'\langle',
    r'\rangle', r'\lfloor', r'\rfloor', r'\lceil', r'\rceil', r'\ulcorner',
    r'\urcorner', r'\lbrack', r'\rbrack'
}
# TODO: looks like left-right do have to match
SIZE_PREFIX = ('left', 'right', 'big', 'Big', 'bigg', 'Bigg')
PUNCTUATION_COMMANDS = {command + opt_space + bracket
                        for command in SIZE_PREFIX
                        for opt_space in {'', ' '}
                        for bracket in BRACKETS_DELIMITERS.union({'|', '.'})}
PUNCTUATION_COMMANDS

In [ ]:
min_example=r"""
\def\bean {\begin{eqnarray*}}  \def\eean {\end{eqnarray*}}
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example))
#print(min_example)
min_example=r"""
the interval $t\in[0,1)$. 
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example))
#print(min_example)
min_example=r"""







\beq
[\chF,\chG\}=\{\partial\chF,\chG\}.
\eeq

derivation $\CA\mapsto [\CB,\CA]$. 







The following characterizations of UAL chains are all equivalent:
\begin{itemize}
    \item[(1)] A skew-symmetric function $\cha:\Lambda^{q+1}\ra\mfkdal$ defines an element of $C_{q}(\mfkdal) $ if $\|\cha\|_{\alpha}<\infty$ for any $\alpha \in \NN$.
    \item[(2)] A skew-symmetric function $\cha:\Lambda^{q+1}\ra\mfkdal$ defines an element of $C_{q}(\mfkdal) $ if there is a function $b(r) \in \Orf$  such that for any $j_0,...,j_q$ the observable $\cha_{j_0...j_q}$ is $b$-localized at $j_a$ for any $a \in \{0,1,...,q\}$.
    \item[(3)] $C_{q}(\mfkdal) $ is the completion of $C_q(\mfkdl) $ with respect to the norms $\|\cdot\|_{\alpha}$.
\end{itemize}
\end{lemma}





""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
\newcommand\const{\operatorname{const}}
""".strip() #.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
\newcommand{\beq}{\begin{equation}}
\newcommand{\eeq}{\end{equation}}
\newcommand{\chF}{{\mathsf f}}
\newcommand{\chG}{{\mathsf g}}
\beq  
[\chF,\chG\}=\{\partial\chF,\chG\}.
\eeq
derivation $\CA\mapsto [\CB,\CA]$. 
""".strip().replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
\[
r_p=d(p,\cdot)\colon \Gamma \to [0,\infty)|~ p \in M\}
\]
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
$\bigl[ a \bigr)$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""

$\varepsilon\in]0,\varepsilon_\star[$,  

""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
\[
i\colon [0,\infty) 
\]
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)

In [ ]:
min_example=r"""
\newcommand\1{{\mathds 1}}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# !! This bug was specific to my fork
min_example=r"""
\newcommand{\linebreakand}{%
    \end{@IEEEauthorhalign}
    \hfill\mbox{}\par
    \mbox{}\hfill\begin{@IEEEauthorhalign}
    }
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
min_example=r"""
 $S \subseteq \{0\} \bigcup [1,\infty) $ if $z^*_2=1$.  
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# two inline math envs next to eachother
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
$\rm{W_{cyc} }\geq 0$$\;\;\square$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
\verb+$TEXMF/tex/latex/elsevier/+, %$%%%%%%%%%%%%%%%%%%%%%%%%%%%%
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# does not handle missing optional braces around arguments
min_example=r"""
$\sqrt {\frac 3 2} >p >1$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
&$\rm{N_{Diskbb}}$$(\times 10^4) $
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
$\frac{j+1+\epsilon}{m^{\alpha}}[$
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
$1\le k< \frac n2 $ 
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
\begin{equation}
\begin{aligned}[t]
[T\tensor*[]{]}{_{\CT}^{\sp}} \\
[T]{_{\CT}^{\sp}}
\end{aligned}
\end{equation}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=1)
#print(min_example)

In [ ]:
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
with open('./data/test.txt', 'r') as infile:
    min_example=infile.read().strip()

TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

In [26]:
# \verb{char}...{char} is also an issue for parser
# !! probably not fixable given the approach used in TexSoup (needs stateful tokenization)
min_example=r"""
\def\f{\frac}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)
import pandas as pd
import numpy as np
pd.DataFrame(np.random.randint(0,100,size=(10, 3)), columns=list('ABC')).to_csv('~/Expire/test_console_upload.csv')

TypeError: [Line: 0, Offset 6] Malformed argument. First and last elements must match a valid argument format. In this case, TexSoup could not find matching punctuation for: {.
Just finished parsing: ['{', TexCmd('frac', [BraceGroup('}')])]

In [26]:
min_example=r"""
\renewcommand{\subsection}[1]{{\textit{#1.~}}}
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

\renewcommand{\subsection}[1]{{\textit{#1.~}}}

In [27]:
min_example=r"""
$\braket{\mathcal N_N^\ell}$

""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
TS.TexSoup(pre_format(min_example), tolerance=0)
#print(min_example)

# The problem the new* changing how to interpret the \begin

$\braket{\mathcal N_N^\ell}$

In [79]:
min_example=r"""
\endinput
""".strip()#.replace('\\}\\', '\\} \\').replace(')}', ') }')
try:
    TS.TexSoup(pre_format(min_example), tolerance=0)
except Exception as e:
    print(e)
    #raise e
#print(min_example)

# The problem the new* changing how to interpret the \begin

'NoneType' object has no attribute 'isdigit'


## Min Example

In [80]:
soup.expr.contents[0].args[2].string

'%\n  First argument: #1 \\\\\n  Second argument: #2 \\\\\n  Repeating first argument: #1 \\\\\n  Repeating second argument: #2\n'

In [90]:
test_string = '%\n {} {##1} First argument: #1 \\\\\n  ##2 Second argument: #2 \\\\\n  Repeating first argument: #1 \\\\\n  Repeating second argument: #2\n'
test_string = ""
pat = re.compile(r"(?<!#)#([0-9]+)")

sub_res = re.sub(pat, lambda match: f"{{arg_{match.group(1)}}}", test_string.replace('{', "{{").replace('}', "}}"))
print(sub_res)
sub_res.format(**{'arg_1':"foo", 'arg_2':'bar'})

#def make_fmt_string(in_string):
    

''

In [92]:
def_fmt_str='\\begin{{eqnarray*}}'
def_fmt_str.format(dict())

'\\begin{eqnarray*}'